<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/Experiment_4_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install hugging face datasets
!pip install datasets
# Importing required libraries
from datasets import load_dataset
import pandas as pd
# Convert trained dataset to pandas DataFrame
raw_dataset = load_dataset("wangrongsheng/ag_news")
df = pd.DataFrame(raw_dataset['train'])
# Printing first 0 rows
print(df.head(10))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
5  Stocks End Up, But Near Year Lows (Reuters) Re...      2
6  Money Funds Fell in Latest Week (AP) AP - Asse...      2
7  Fed minutes show dissent over inflation (USATO...      2
8  Safety Net (Forbes.com) Forbes.com - After ear...      2
9  Wall St. Bears Claw Back Into the Black  NEW Y...      2


In [2]:
# import regular expression
import re

# Function for clean the data
def clean_text(text):

    text = text.lower()                 # Convert to lowercase
    text = re.sub(r'[^a-zA-Z ]', '', text)  # Remove numbers & special characters
    return text
# Apply the cleaning function to every sample text
df["text"] = df["text"].apply(clean_text)
# Print first 5 rows of the cleaned dataset
print(df["text"].head())

0    wall st bears claw back into the black reuters...
1    carlyle looks toward commercial aerospace reut...
2    oil and economy cloud stocks outlook reuters r...
3    iraq halts oil exports from main southern pipe...
4    oil prices soar to alltime record posing new m...
Name: text, dtype: object


In [3]:
# import the tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
# set maximum vocabulary size
vocab_size = 10000
# to create object for the tokenizer
tokenizer = Tokenizer(num_words=vocab_size)
# Building vocabulary from the text
tokenizer.fit_on_texts(df["text"])
# Convert every sentence into sequence of integer
sequences = tokenizer.texts_to_sequences(df["text"])

In [4]:
# Import pad_Sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Fixing sequence length
max_length = 50
# Making Every sentence exactly 50 words
X = pad_sequences(sequences,
                  maxlen=max_length,
                  padding="post",
                  truncating="post")

print(X.shape)

(120000, 50)


In [5]:
# import one-hot encoding
from tensorflow.keras.utils import to_categorical
# convert labels into one-hot encoded vectors
y = to_categorical(df["label"])

print(y[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [6]:
# import required keras classes
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
# create sequential model
model = Sequential()
# add embedding layer
model.add(Embedding(vocab_size, 64, input_length=max_length))
# add rnn layer with 64 neurons
model.add(SimpleRNN(64))
# 4 classes, sofftmax activation
model.add(Dense(4, activation="softmax"))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Compile and train the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"])
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.7754 - loss: 0.6100 - val_accuracy: 0.8080 - val_loss: 0.5184
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8585 - loss: 0.4273 - val_accuracy: 0.8408 - val_loss: 0.4689
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8766 - loss: 0.3816 - val_accuracy: 0.8553 - val_loss: 0.4434
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.8956 - loss: 0.3369 - val_accuracy: 0.8478 - val_loss: 0.4434
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9016 - loss: 0.3209 - val_accuracy: 0.8624 - val_loss: 0.4367


In [9]:
#Evaluating the Accuracy of the Trained model
loss, accuracy = model.evaluate(X, y)

print("Accuracy:", accuracy)

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8916 - loss: 0.3486
Accuracy: 0.8915500044822693
